In [9]:
import json
from pathlib import Path

import torch
from loguru import logger
from tqdm import tqdm
import typer

from gelos.gelosdatamodule import GELOSDataModule
from src.gelosdataset_lc import GELOSLCDataSet

In [12]:
data_version = 'v0.50.1'
data_root =  Path('/app/data/raw')
output_dir = data_root
batch_size = 8
num_workers = 16

In [13]:
data_root = data_root / data_version
output_path = output_dir / data_version / "statistics.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

logger.info(f"Data root: {data_root}")
logger.info(f"Output path: {output_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

datamodule = GELOSDataModule(
    data_root=data_root,
    dataset_class=GELOSLCDataSet,
    batch_size=batch_size,
    num_workers=num_workers,
)
datamodule.setup("predict")
loader = datamodule.predict_dataloader()
dataset = loader.dataset

modalities = list(dataset.bands.keys())

2026-02-19 23:43:39.542 | INFO     | __main__:<module>:5 - Data root: /app/data/raw/v0.50.1
2026-02-19 23:43:39.544 | INFO     | __main__:<module>:6 - Output path: /app/data/raw/v0.50.1/statistics.json
2026-02-19 23:43:39.546 | INFO     | __main__:<module>:9 - Using device: cpu


In [21]:
# Initialize accumulators
sums = {}
sum_squares = {}
pixel_counts = {}
first_batch = True

for batch in tqdm(loader, total=len(loader), desc="Computing statistics"):
    image_dict = batch["image"]

    for modality, tensor in image_dict.items():
        tensor = tensor.to(device)

        if first_batch:
            num_channels = tensor.shape[1]
            sums[modality] = torch.zeros(num_channels, device=device)
            sum_squares[modality] = torch.zeros(num_channels, device=device)
            _, t, _, h, w = tensor.shape
            pixel_counts[modality] = len(dataset) * t * h * w

        sums[modality] += torch.sum(tensor, dim=(0, 2, 3, 4))
        sum_squares[modality] += torch.sum(tensor.pow(2), dim=(0, 2, 3, 4))
    break

    if first_batch:
        first_batch = False


Computing statistics:   0%|          | 0/9824 [00:02<?, ?it/s]


In [22]:
for source, tensor in image_dict.items():
    print(source, tensor.shape)

S1RTC torch.Size([8, 2, 4, 96, 96])
S2L2A torch.Size([8, 12, 4, 96, 96])
LC2L2 torch.Size([8, 7, 4, 32, 32])
DEM torch.Size([8, 1, 1, 32, 32])


In [ ]:
print(sums)

{'S1RTC': tensor([68945.5312, 16372.8027]),
 'S2L2A': tensor([3.6145e+08, 3.7226e+08, 4.1269e+08, 3.7314e+08, 4.8230e+08, 8.6869e+08,
         1.0385e+09, 1.0479e+09, 1.1204e+09, 1.1163e+09, 6.9138e+08, 4.7098e+08]),
 'LC2L2': tensor([ 493.9813,  732.5305, 1438.6884, 1041.7100, 9173.8535, 4247.0278,
         1812.6193]),
 'DEM': tensor([3960248.])}

In [ ]:
# Initialize accumulators
sums = {}
sum_squares = {}
pixel_counts = {}
first_batch = True

for batch in tqdm(loader, total=len(loader), desc="Computing statistics"):
    image_dict = batch["image"]

    for modality, tensor in image_dict.items():
        tensor = tensor.to(device)

        if first_batch:
            num_channels = tensor.shape[1]
            sums[modality] = torch.zeros(num_channels, device=device)
            sum_squares[modality] = torch.zeros(num_channels, device=device)
            _, t, _, h, w = tensor.shape
            pixel_counts[modality] = len(dataset) * t * h * w

        sums[modality] += torch.sum(tensor, dim=(0, 2, 3, 4))
        sum_squares[modality] += torch.sum(tensor.pow(2), dim=(0, 2, 3, 4))

    if first_batch:
        first_batch = False

# Compute means and stds
means = {m: sums[m] / pixel_counts[m] for m in modalities}
stds = {m: torch.sqrt(sum_squares[m] / pixel_counts[m] - means[m].pow(2)) for m in modalities}

# Format results using band names from the dataset class
bands_per_modality = dataset.all_band_names
formatted_means = {}
formatted_stds = {}

for modality in modalities:
    mean_values = means[modality].cpu().tolist()
    std_values = stds[modality].cpu().tolist()
    band_names = bands_per_modality[modality]

    formatted_means[modality] = {band: mean for band, mean in zip(band_names, mean_values)}
    formatted_stds[modality] = {band: std for band, std in zip(band_names, std_values)}

results = {"MEANS": formatted_means, "STDS": formatted_stds}

logger.info("MEANS = " + json.dumps(formatted_means, indent=4))
logger.info("STDS = " + json.dumps(formatted_stds, indent=4))

with open(output_path, "w") as f:
    json.dump(results, f, indent=4)
logger.info(f"Saved statistics to {output_path}")


Computing statistics:   0%|          | 0/9824 [00:02<?, ?it/s]
2026-02-19 23:47:50.900 | INFO     | __main__:<module>:45 - MEANS = {
    "S1RTC": {
        "VV": 4.759857984026894e-05,
        "VH": 1.1303447536192834e-05
    },
    "S2L2A": {
        "COASTAL_AEROSOL": 0.04158959165215492,
        "BLUE": 0.042833466082811356,
        "GREEN": 0.04748561233282089,
        "RED": 0.042934600263834,
        "RED_EDGE_1": 0.05549539625644684,
        "RED_EDGE_2": 0.09995470941066742,
        "RED_EDGE_3": 0.11949052661657333,
        "NIR_BROAD": 0.12056981772184372,
        "NIR_NARROW": 0.1289205402135849,
        "WATER_VAPOR": 0.12844201922416687,
        "SWIR_1": 0.07955193519592285,
        "SWIR_2": 0.05419204756617546
    },
    "LC2L2": {
        "coastal": 8.769459327595541e-07,
        "blue": 1.3004331549382186e-06,
        "green": 2.554048023739597e-06,
        "red": 1.8493076368031325e-06,
        "nir08": 1.6285988749586977e-05,
        "swir16": 7.5395851126813795e-06

PermissionError: [Errno 13] Permission denied: '/app/data/raw/v0.50.1/statistics.json'